# Collected dataset

This notebook reproduces the released-corpus counts in §3.1–§3.2 directly from `raw/`.

In [1]:
from pathlib import Path
import json, re
from collections import Counter, defaultdict
from urllib.parse import urlparse

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "raw").is_dir():
    ROOT = ROOT.parent
RAW_AIOS = ROOT / "raw" / "aio_results"

CATEGORY_NAMES = {
    "autos_vehicles": "Autos & Vehicles", "beauty_fashion": "Beauty & Fashion",
    "business_finance": "Business & Finance", "climate": "Climate",
    "entertainment": "Entertainment", "food_drink": "Food & Drink",
    "games": "Games", "health": "Health", "hobbies_leisure": "Hobbies & Leisure",
    "jobs_education": "Jobs & Education", "law_gov": "Law & Government",
    "other": "Other", "pets_animals": "Pets & Animals", "politics": "Politics",
    "science": "Science", "shopping": "Shopping", "sports": "Sports",
    "technology": "Technology", "travel_transport": "Travel & Transport",
}

def category_from_path(path, root=RAW_AIOS):
    folder = path.relative_to(root).parts[1]
    slug = re.match(r"trending_US_1d_(.+?)_\d{4}-", folder).group(1)
    return CATEGORY_NAMES[slug]

def hostname(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def show_table(frame, formats=None):
    output = frame.copy()
    for column, template in (formats or {}).items():
        output[column] = output[column].map(template.format)
    display(HTML(output.to_html(index=False, border=0)))
import hashlib

RAW_TRENDS = ROOT / "raw" / "trends"
RAW_QUERIES = ROOT / "raw" / "query_results"

trend_files = list(RAW_TRENDS.rglob("*.csv"))
query_files = list(RAW_QUERIES.rglob("result.json"))
aio_files = list(RAW_AIOS.rglob("result.json"))

days = {re.search(r"(2026-\d{2}-\d{2})T", path.parts[-2]).group(1) for path in trend_files}
categories = {re.match(r"trending_US_1d_(.+?)_2026-", path.name).group(1) for path in trend_files}

reference_count = comparison_aios = comparison_references = serp_links = 0
hosts = set()
missing_page_captures = 0
for path in aio_files:
    record = json.loads(path.read_text(encoding="utf-8"))
    references = (record.get("aiOverview") or {}).get("references") or []
    urls = [item.get("url") for item in references if isinstance(item, dict) and item.get("url")]
    reference_count += len(urls)
    hosts.update(hostname(url) for url in urls if hostname(url))

    page_dir = path.parent / "page_analysis"
    for url in urls:
        page_id = hashlib.md5(url.encode("utf-8")).hexdigest()[:8]
        missing_page_captures += not (page_dir / f"{page_id}.json").is_file()

    serp = [url for url in (record.get("serpLinksDedup") or []) if isinstance(url, str)]
    has_goto = any(hostname(url) == "google.com" and urlparse(url).path.rstrip("/") == "/goto" for url in serp)
    if (path.parent / "snapshot.html").is_file() and not has_goto:
        comparison_aios += 1
        comparison_references += len(urls)
        serp_links += len(serp)

assert len(days) == 40 and len(categories) == 19 and len(trend_files) == 760
assert len(query_files) == 55_393 and len(aio_files) == 7_583
assert reference_count == 61_212 and len(hosts) == 7_479
assert (comparison_aios, comparison_references, serp_links) == (7_562, 61_046, 241_710)
assert missing_page_captures == 44

summary = pd.DataFrame([
    ["Observation window", f"{min(days)} to {max(days)}"],
    ["Days", len(days)], ["Google Trends categories", len(categories)],
    ["Google Trends CSVs", len(trend_files)], ["Query executions", len(query_files)],
    ["AIO observations", len(aio_files)], ["AIO activation", f"{len(aio_files)/len(query_files):.1%}"],
    ["AIO reference occurrences", reference_count], ["Unique AIO hostnames", len(hosts)],
    ["AIO/SERP comparison cohort", comparison_aios],
    ["AIO references in comparison cohort", comparison_references],
    ["First-page SERP links", serp_links],
    ["References without a page capture", f"{missing_page_captures} ({missing_page_captures/reference_count:.2%})"],
], columns=["Measure", "Value"])
show_table(summary)

Measure,Value
Observation window,2026-03-13 to 2026-04-21
Days,40
Google Trends categories,19
Google Trends CSVs,760
Query executions,55393
AIO observations,7583
AIO activation,13.7%
AIO reference occurrences,61212
Unique AIO hostnames,7479
AIO/SERP comparison cohort,7562
